# Audio Feature Extraction with librosa

**CS 89.02 / MUS 14.05 — Music and AI, Week 4**

This notebook introduces the core audio features used in Music Information Retrieval (MIR).
We will extract and visualize:

- **Waveform and Mel Spectrogram** — time-domain and time-frequency representations
- **MFCCs** — compact timbral descriptors widely used in speech and music
- **Chroma features** — pitch-class energy profiles capturing harmonic content
- **Onset detection** — locating note and beat attacks
- **Tempo and beat tracking** — rhythmic structure estimation

These features form the building blocks for classification, retrieval, and generative models.

In [ ]:
!pip install librosa matplotlib numpy

In [ ]:
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd

# Load a built-in example (Brahms Hungarian Dance No. 5)
filename = librosa.example('brahms')
y, sr = librosa.load(filename, duration=30)  # load 30 seconds

print(f"Sample rate: {sr} Hz")
print(f"Duration: {len(y) / sr:.2f} seconds")
print(f"Samples: {len(y)}")

# Listen to the audio
ipd.Audio(y, rate=sr)

## Waveform and Spectrogram

The **waveform** shows amplitude over time — useful for seeing dynamics and silence,
but it tells us little about pitch or timbre.

The **mel spectrogram** applies the Short-Time Fourier Transform (STFT) and maps
frequencies onto the mel scale, which approximates human pitch perception.
Low frequencies get finer resolution; high frequencies are grouped more coarsely.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Waveform
librosa.display.waveshow(y, sr=sr, ax=axes[0], alpha=0.7)
axes[0].set_title('Waveform')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')

# Mel spectrogram
S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmax=8000)
S_dB = librosa.power_to_db(S, ref=np.max)

img = librosa.display.specshow(S_dB, x_axis='time', y_axis='mel',
                               sr=sr, fmax=8000, ax=axes[1])
axes[1].set_title('Mel Spectrogram')
fig.colorbar(img, ax=axes[1], format='%+2.0f dB')

plt.tight_layout()
plt.show()

## MFCCs (Mel-Frequency Cepstral Coefficients)

MFCCs are the most widely used features in audio classification. They are computed by:

1. Computing the mel spectrogram
2. Taking the log of the mel energies
3. Applying the Discrete Cosine Transform (DCT)

The first ~13 coefficients capture the **spectral envelope** — a compact description
of timbre. MFCC 0 relates to overall energy; higher coefficients capture increasingly
fine spectral detail.

MFCCs discard fine pitch information (the harmonic structure) and retain the
"shape" of the spectrum — which is why they work well for timbre-based tasks
(genre, instrument, speaker ID) but poorly for melody or harmony tasks.

In [ ]:
# Extract 13 MFCCs
mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

print(f"MFCC shape: {mfccs.shape}  (n_mfcc x time_frames)")

fig, ax = plt.subplots(figsize=(12, 4))
img = librosa.display.specshow(mfccs, x_axis='time', ax=ax, sr=sr)
ax.set_title('MFCCs (13 coefficients)')
ax.set_ylabel('MFCC Index')
fig.colorbar(img, ax=ax)
plt.tight_layout()
plt.show()

## Chroma Features

Chroma features (also called a **chromagram**) project the spectrum onto the
12 pitch classes (C, C#, D, ..., B), summing energy across all octaves.

This representation captures **harmonic and melodic content** while being
invariant to octave — a C4 and a C5 both contribute to the same chroma bin.

Chroma features are useful for:
- Chord recognition
- Key estimation
- Cover song identification
- Harmonic similarity

In [ ]:
# Extract chroma features
chroma = librosa.feature.chroma_stft(y=y, sr=sr)

print(f"Chroma shape: {chroma.shape}  (12 pitch classes x time_frames)")

fig, ax = plt.subplots(figsize=(12, 4))
img = librosa.display.specshow(chroma, y_axis='chroma', x_axis='time', ax=ax, sr=sr)
ax.set_title('Chromagram')
fig.colorbar(img, ax=ax)
plt.tight_layout()
plt.show()

## Onset Detection

Onset detection identifies the moments when new notes or events begin.
librosa computes an **onset strength envelope** (measuring spectral flux)
and then picks peaks as onset times.

Onsets are fundamental to rhythm analysis, beat tracking, and segmentation.

In [ ]:
# Detect onsets
onset_frames = librosa.onset.onset_detect(y=y, sr=sr)
onset_times = librosa.frames_to_time(onset_frames, sr=sr)
onset_env = librosa.onset.onset_strength(y=y, sr=sr)

print(f"Detected {len(onset_times)} onsets")

fig, ax = plt.subplots(figsize=(12, 4))
times = librosa.times_like(onset_env, sr=sr)
ax.plot(times, onset_env, label='Onset strength')
ax.vlines(onset_times, 0, onset_env.max(), color='r', alpha=0.5,
          linestyle='--', label='Onsets')
ax.set_title('Onset Detection')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Onset Strength')
ax.legend()
plt.tight_layout()
plt.show()

## Tempo and Beat Tracking

Beat tracking estimates the **tempo** (BPM) and the **beat positions** in the audio.
librosa uses the onset strength envelope to find a periodicity via autocorrelation,
then applies dynamic programming to place beats at consistent intervals.

In [ ]:
# Estimate tempo and beat positions
tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr)
beat_times = librosa.frames_to_time(beat_frames, sr=sr)

print(f"Estimated tempo: {tempo:.1f} BPM")
print(f"Number of beats: {len(beat_times)}")

fig, ax = plt.subplots(figsize=(12, 4))
librosa.display.waveshow(y, sr=sr, ax=ax, alpha=0.5)
ax.vlines(beat_times, -1, 1, color='r', alpha=0.7, linestyle='-', label='Beats')
ax.set_title(f'Beat Tracking (tempo = {tempo:.1f} BPM)')
ax.set_xlabel('Time (s)')
ax.legend()
plt.tight_layout()
plt.show()

## Feature Summary

For classification, we typically reduce time-varying features to **summary statistics**
(mean, standard deviation) across the entire clip. This converts a variable-length
audio signal into a fixed-length feature vector suitable for classifiers like SVM.

The function below extracts a comprehensive feature vector from any audio signal.

In [ ]:
def extract_feature_summary(y, sr):
    """Extract a fixed-length feature vector from an audio signal.

    Returns a dict of feature names -> values, suitable for
    building a feature matrix for classification.
    """
    features = {}

    # MFCCs (13 coefficients x 2 stats = 26 features)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    for i in range(13):
        features[f'mfcc_{i}_mean'] = np.mean(mfccs[i])
        features[f'mfcc_{i}_std'] = np.std(mfccs[i])

    # Chroma (12 pitch classes x 2 stats = 24 features)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    for i in range(12):
        features[f'chroma_{i}_mean'] = np.mean(chroma[i])
        features[f'chroma_{i}_std'] = np.std(chroma[i])

    # Spectral features (2 stats each = 8 features)
    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    features['spectral_centroid_mean'] = np.mean(spectral_centroid)
    features['spectral_centroid_std'] = np.std(spectral_centroid)

    spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    features['spectral_bandwidth_mean'] = np.mean(spectral_bandwidth)
    features['spectral_bandwidth_std'] = np.std(spectral_bandwidth)

    spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
    features['spectral_rolloff_mean'] = np.mean(spectral_rolloff)
    features['spectral_rolloff_std'] = np.std(spectral_rolloff)

    zcr = librosa.feature.zero_crossing_rate(y)[0]
    features['zcr_mean'] = np.mean(zcr)
    features['zcr_std'] = np.std(zcr)

    # Tempo
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    features['tempo'] = tempo

    # RMS energy
    rms = librosa.feature.rms(y=y)[0]
    features['rms_mean'] = np.mean(rms)
    features['rms_std'] = np.std(rms)

    return features


# Extract and display
summary = extract_feature_summary(y, sr)
print(f"Total features: {len(summary)}")
print(f"\nFeature vector:")
for name, value in list(summary.items())[:10]:
    print(f"  {name:30s} = {value:.4f}")
print(f"  ... and {len(summary) - 10} more")